In [ ]:
import os
import glob

In [ ]:
def read_files(path):
    file_list = glob.glob(os.path.join(path, "*.txt"))
    
    sentences = []
    for file in file_list:
        with open(file, 'r', encoding='utf-8') as f:
            sentences.extend(f.readlines())
    
    return sentences

In [ ]:
base_path = "./data/"
training_path = os.path.join(base_path, "training")
name = os.listdir(training_path)

In [ ]:
path = [os.path.join(training_path, name[i]).replace("\\", "/") for i in range(1, 8)]

In [ ]:
crime_t = read_files(path[0])
blame_t = read_files(path[1])
chose_t = read_files(path[2])
ad_t = read_files(path[3])
fuck_t = read_files(path[4])
violence_t = read_files(path[5])
rumor_t = read_files(path[6])

In [ ]:
validation_path = os.path.join(base_path, "validation")
name = os.listdir(validation_path)

In [ ]:
path = [os.path.join(validation_path, name[i]) for i in range(1, 8)] 

In [ ]:
crime_v = read_files(path[0])
blame_v = read_files(path[1])
chose_v = read_files(path[2])
ad_v = read_files(path[3])
fuck_v = read_files(path[4])
violence_v = read_files(path[5])
rumor_v = read_files(path[6])

In [ ]:
train = sum([crime_t, blame_t, chose_t, ad_t, fuck_t, violence_t, rumor_t], [])
valid = sum([crime_v, blame_v, chose_v, ad_v, fuck_v, violence_v, rumor_v], [])

In [ ]:
print(len(valid))
print(len(train))

In [ ]:
!pip install transformers
!pip install torch

In [ ]:
pip install kobert-transformers

In [ ]:
import torch
from transformers import BertModel, BertTokenizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

from transformers import BertTokenizer
from kobert_transformers import get_tokenizer

In [ ]:
tokenizer = get_tokenizer()
model = BertModel.from_pretrained('skt/kobert-base-v1')

# GPU 사용 가능 시 설정 (없으면 CPU 사용)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# 아래 뜨는거 미래 경고임 없애고 싶으면 torenizer 함수 쓸 때, clean_up_tokenization_spaces=False 인수 추가  

In [ ]:
# 문장 리스트
sentences = valid

# 각 문장의 임베딩 구하기
def get_embeddings(sentences):
    embeddings = []
    for sentence in sentences:
        inputs = tokenizer(sentence, return_tensors='pt', padding=True, truncation=True).to(device)
        with torch.no_grad():
            outputs = model(**inputs)
            sentence_embedding = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
            embeddings.append(sentence_embedding)
    return np.vstack(embeddings)

# 문장 임베딩
sentence_embeddings = get_embeddings(sentences)
print(f"Sentence Embeddings Shape: {sentence_embeddings.shape}")

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
def soft_cosine_similarity(embeddings, word_similarity_matrix):
    num_sentences = embeddings.shape[0]
    scm_matrix = np.zeros((num_sentences, num_sentences))

    for i in range(num_sentences):
        for j in range(num_sentences):
            A = embeddings[i].reshape(1, -1)
            B = embeddings[j].reshape(1, -1)

            # Numerator: A^T S B
            numerator = np.dot(np.dot(A, word_similarity_matrix), B.T)

            # Denominator: sqrt(A^T S A) * sqrt(B^T S B)
            denominator = np.sqrt(np.dot(np.dot(A, word_similarity_matrix), A.T)) * np.sqrt(np.dot(np.dot(B, word_similarity_matrix), B.T))

            # SCM calculation
            scm_matrix[i, j] = (numerator / denominator).item()

    return scm_matrix


In [ ]:
# 코사인 유사도 계산
word_similarity_matrix = cosine_similarity(sentence_embeddings.T)  # 단어 간 유사도 행렬 생성
scm_matrix = soft_cosine_similarity(sentence_embeddings, word_similarity_matrix)

# 유사도 행렬 출력
print("Soft Cosine Similarity Matrix:")
print(scm_matrix)

In [ ]:
scm_matrix[0:5,1000:1005]

In [ ]:
import numpy as np

def five_number_summary(matrix):
    summary = {}
    # 각 열에 대해 다섯 수치 요약 계산
    summary['min'] = np.min(matrix)
    summary['Q1'] = np.percentile(matrix, 25)
    summary['median'] = np.median(matrix)
    summary['Q3'] = np.percentile(matrix, 75)
    summary['max'] = np.max(matrix)
    
    return summary

summary = five_number_summary(scm_matrix)

# 다섯 수치 요약 출력
for stat, values in summary.items():
    print(f"{stat}: {values}")

In [ ]:
sentence_embeddings[0,]